## RA Coding Drills
#### Gavin Qu 12-18-2025

**Useful pattern:** 
``cols = [c for c in target if c in df.columns] df_subset = df[cols].copy()``

### Day 12 - Review + Extension
**Dataset:** UCI Adult Income
### Drill 1 - Data Audit and Encoding

In [6]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import os

# fetch
census_income = fetch_ucirepo(id=20)

print(census_income.metadata)
print(census_income.variables)

{'uci_id': 20, 'name': 'Census Income', 'repository_url': 'https://archive.ics.uci.edu/dataset/20/census+income', 'data_url': 'https://archive.ics.uci.edu/static/public/20/data.csv', 'abstract': 'Predict whether income exceeds $50K/yr based on census data.  Also known as Adult dataset.', 'area': 'Social Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 48842, 'num_features': 14, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Income', 'Education Level', 'Other', 'Race', 'Sex'], 'target_col': ['income'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1996, 'last_updated': 'Mon Sep 09 2024', 'dataset_doi': '10.24432/C5GP7S', 'creators': ['Ron Kohavi'], 'intro_paper': None, 'additional_info': {'summary': 'Extraction was done by Barry Becker from the 1994 Census database.  A set of reasonably clean records was extracted using the following conditions: ((AAGE>16) && 

In [7]:
X = census_income.data.features
y = census_income.data.targets

df = pd.concat([X, y], axis=1)

print(f"Shape of data: {df.shape}\n")
print(f"Data info: {df.info()}\n")
print(f"Summary Stats: \n{df.describe().T}")

Shape of data: (48842, 15)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             48842 non-null  int64 
 1   workclass       47879 non-null  object
 2   fnlwgt          48842 non-null  int64 
 3   education       48842 non-null  object
 4   education-num   48842 non-null  int64 
 5   marital-status  48842 non-null  object
 6   occupation      47876 non-null  object
 7   relationship    48842 non-null  object
 8   race            48842 non-null  object
 9   sex             48842 non-null  object
 10  capital-gain    48842 non-null  int64 
 11  capital-loss    48842 non-null  int64 
 12  hours-per-week  48842 non-null  int64 
 13  native-country  48568 non-null  object
 14  income          48842 non-null  object
dtypes: int64(6), object(9)
memory usage: 5.6+ MB
Data info: None

Summary Stats: 
                  count         

In [8]:
df.head(1)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K


`fnlwgt` (Final Weight) is a crucial feature in the UCI Adult Census Income dataset (and similar Census data), representing the number of people in the U.S. population that a single data record actually stands for, helping to scale samples to the entire population for accurate analysis, especially for predicting income.  

1. Load data and perform a 60-second audit:
    * .info(), .describe(include="all")
    * Identify at least 3 data quality issues (types, missingness, encoding).
2. Convert all categorical variables to category.
3. Create a clean income_binary variable (0/1).

In [9]:
# catch encoding issue
non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"non-numeric columns: {non_numeric}\n")
# find missing values by col
missing_pct = df.isna().mean() * 100
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print(f"\nUnique genders: {df['sex'].unique()}")

non-numeric columns: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']

occupation        1.977806
workclass         1.971664
native-country    0.560993
dtype: float64

Unique genders: ['Male' 'Female']


In [10]:
# convert dtype to category
obj_cols = df.select_dtypes(include=['object']).columns
df[obj_cols] = df[obj_cols].astype('category')
# create clean income binary var
df['income'] = df['income'].str.strip().str.replace('.', '', regex=False) # replace trailing . with ''
df['income_high'] = (df['income'] == '>50K').astype(int)
print(df.groupby('income')['income_high'].value_counts())

income  income_high
<=50K   0              37155
>50K    1              11687
Name: count, dtype: int64


The income data is converted to a binary value with either high income or not. Missingness is likely not random as people in certain class of features may not report income, this could show up as a confounder later on. High earners and low earners at extreme ends may not want to report income. We can test if the missing pattern correlates with other known covariates. However, there is no data that supports that missingness is not random and the income feature only has negligible missing data. 

### Drill 2 - GroupBy + Conditional Aggregation (~30 min)
**Tasks**
1. Compute mean hours_per_week by:
    * education
    * sex × education
2. For each education level, compute:
    * Share earning >50K
    * Sample size
3. Drop education groups with fewer than 50 observations.
4. Rank remaining education levels by income share (descending).
5. Markdown cell answering:
    * Why is dropping small groups preferable to keeping them in this context?

In [30]:
mean_hours_by_edu = df.groupby('education')['hours-per-week'].mean()
mean_hours_by_edu_sex = df.groupby(['sex','education'])['hours-per-week'].mean()

share_earning = df.groupby('education').agg(
    share_high_income=('income_high', 'mean'),
    sample_size=('income_high','size')
)
filtered = df.groupby('education').filter(lambda x: len(x) >= 100)

ranking = filtered.groupby('education')['income_high'].mean().sort_values(ascending=False)
print(f"ranking of income share by education level: {ranking}")

ranking of income share by education level: education
Prof-school     0.739808
Doctorate       0.725589
Masters         0.549116
Bachelors       0.412835
Assoc-acdm      0.257964
Assoc-voc       0.253275
Some-college    0.189649
HS-grad         0.158578
12th            0.073059
7th-8th         0.064921
10th            0.062635
9th             0.054233
5th-6th         0.053045
11th            0.050773
1st-4th         0.032389
Preschool            NaN
Name: income_high, dtype: float64


/var/folders/n2/8hz3y3r90rj63gkzgrl1hwg40000gn/T/ipykernel_31725/3862499958.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_hours_by_edu = df.groupby('education')['hours-per-week'].mean()
/var/folders/n2/8hz3y3r90rj63gkzgrl1hwg40000gn/T/ipykernel_31725/3862499958.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_hours_by_edu_sex = df.groupby(['sex','education'])['hours-per-week'].mean()
/var/folders/n2/8hz3y3r90rj63gkzgrl1hwg40000gn/T/ipykernel_31725/3862499958.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass obser

Answer: dropping groups with small counts are preferable because it will affect the distributions to create outliers and noise. 

In [32]:
df.head(1)

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,income_high
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,0


### Drill 3 — Statistical Testing & Uncertainty (≈30 min)
**Tasks**
1. Split the sample into:
    * income >50K
    * income ≤50K
2. Compute the difference in mean hours_per_week.
3. Conduct:
    * Two-sample t-test
    * Bootstrap confidence interval for the mean difference

In [39]:
mask = df['income'] == '>50K'

high_income_hours = df[mask]['hours-per-week']
low_income_hours = df[~mask]['hours-per-week']

diff_in_mean_hours = high_income_hours.mean() - low_income_hours.mean()
print(f"difference in mean working hours is {diff_in_mean_hours:.2f} hours")

# two sample t-test: mean, variance and size for income variables
x1_mean = high_income_hours.mean()
x2_mean = low_income_hours.mean()
x1_var = high_income_hours.var()
x2_var = low_income_hours.var()
n1 = high_income_hours.count()
n2 = low_income_hours.count()

t_score = (x1_mean - x2_mean) / np.sqrt((x1_var / n1) + (x2_var / n2))
print(f"t-test score is {t_score}")

# bootstrap CI for mean diff
iterations = 1000
n_high = len(high_income_hours)
n_low = len(low_income_hours)

indices_low = np.random.randint(0, n_low, size=(iterations, n_low))
indices_high = np.random.randint(0, n_high, size=(iterations, n_high))

low_income_array = low_income_hours.values
high_income_array = high_income_hours.values

boot_samples_low = low_income_array[indices_low]
boot_samples_high = high_income_array[indices_high]

boot_mean_high = boot_samples_high.mean(axis=1)
boot_mean_low = boot_samples_low.mean(axis=1)
boot_diff = boot_mean_high - boot_mean_low
ci_high = np.percentile(boot_diff, [2.5,97.5])
print(f"95% confidence interval is {ci_high}")

difference in mean working hours is 6.61 hours
t-test score is 54.662247230930255
95% confidence interval is [6.39323351 6.84381876]


### Drill 4 — Regression & Interpretation (≈30 min)
**Tasks**
1. Fit OLS:
`income_binary ~ education_num + age + hours_per_week + sex`
2. Add occupation controls and refit.
3. Standardize numeric covariates and refit again.
4. Compare coefficients across specifications.

In [52]:
import statsmodels.formula.api as smf

model1 = smf.ols(
    formula='income_high ~ Q("education-num") + age + Q("hours-per-week") + sex',   # Q("") since Pasty treats hyphens as minus sign
    data=df
).fit()
print(model1.summary())

                            OLS Regression Results                            
Dep. Variable:            income_high   R-squared:                       0.212
Model:                            OLS   Adj. R-squared:                  0.212
Method:                 Least Squares   F-statistic:                     3294.
Date:                Mon, 22 Dec 2025   Prob (F-statistic):               0.00
Time:                        18:19:16   Log-Likelihood:                -21867.
No. Observations:               48842   AIC:                         4.374e+04
Df Residuals:                   48837   BIC:                         4.379e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              -0.7927    